# Phase 1 — exact 108-route Colab capture

Target: OLMoE 45, Inkling 21, Qwen4 21, GLM5-Next 21. A route counts only when its exact receipt passes the existing gate. Other admitted Phase 1 routes remain outside this target. Supply real checkpoints for each identity and storage encoding, media, and OLMoE manifests. Required host/device memory depends on the actual checkpoints; no universal T4 fit is assumed.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/openagent')
if not ROOT.exists():
    with zipfile.ZipFile('/content/drive/MyDrive/openagent-phase1.zip') as archive:
        for member in archive.infolist():
            if not (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()) or ((member.external_attr>>16)&0o170000)==0o120000:
                raise ValueError('Unsafe ZIP path or symbolic link')
        archive.extractall(ROOT)
if not (ROOT/'phase1_verify.py').is_file():
    raise FileNotFoundError('The working-tree ZIP must contain repository files at its root')
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-e','.', 'numpy','pillow','torch','safetensors','tokenizers','transformers==5.16.1'],check=True)


In [ ]:
from openagent.adaptive_engine.native_family_capture import compile_native_text_kernel,compile_native_source
from openagent.adaptive_engine.device_capture import compile_segment_executor,compile_segment_runtime
BUILD=ROOT/'build/colab-capture'
libraries={family:str(compile_native_text_kernel(family,BUILD/family)) for family in ('inkling','qwen4','glm5-next')}
libraries.update(media=str(compile_native_source('native_media_ops.c','media-ops',BUILD/'media')),
                 olmoe=str(compile_segment_executor(BUILD/'olmoe')),
                 segment=str(compile_segment_runtime(BUILD/'segment')))
print(json.dumps(libraries,indent=2))


## Bind actual artifacts

On its first run, the next cell writes an asset template to Drive. Fill every checkpoint and manifest path and supply one real image. Text and multimodal identities need their corresponding checkpoint configs. Selecting F16/BF16/FP8 does not convert an F32 checkpoint. Preflight rejects mismatched identities, encodings, missing files and invalid CLI arguments before capture.

In [ ]:
from openagent.adaptive_engine.phase1_colab import target_rows
ASSETS=Path('/content/drive/MyDrive/phase1-colab-assets.json')
OUT=Path('/content/drive/MyDrive/phase1-capture')
OUT.mkdir(parents=True,exist_ok=True)
if not ASSETS.is_file():
    checkpoints={}
    for row in target_rows():
        checkpoints.setdefault(row['model_type'],{})[row['source_format']]='/content/drive/MyDrive/checkpoints/'+row['model_type']+'/'+row['source_format']
    template=dict(schema='openagent.phase1-colab-assets.v1',checkpoints=checkpoints,
        manifests={row['persisted_format']:'/content/drive/MyDrive/manifests/'+row['persisted_format']+'.json' for row in target_rows() if row['family']=='olmoe'},
        prompts={},image='/content/drive/MyDrive/capture-input.png',device='cuda:0',max_tokens=4,context_tokens=512)
    ASSETS.write_text(json.dumps(template,indent=2))
    raise RuntimeError('Fill real artifact paths in '+str(ASSETS)+', then rerun this cell')
assets=json.loads(ASSETS.read_text())
assets.update(libraries=libraries,receipts=str(OUT/'phase1_runtime_receipts.json'),evidence_root=str(OUT/'evidence'))
from transformers import AutoProcessor
for identity in ('inkling_mm_model','qwen4_exp','glm5_next'):
    if not assets.setdefault('prompts',{}).get(identity):
        processor=AutoProcessor.from_pretrained(assets['checkpoints'][identity]['F32'],local_files_only=True,trust_remote_code=False)
        assets['prompts'][identity]=processor.apply_chat_template([{'role':'user','content':[{'type':'image'},{'type':'text','text':'Describe this image.'}]}],tokenize=False,add_generation_prompt=True)
ASSETS.write_text(json.dumps(assets,indent=2))


In [ ]:
PLAN=OUT/'capture-plan.json'
base=[sys.executable,'-m','openagent.adaptive_engine.phase1_colab']
subprocess.run(base+['plan','--assets',str(ASSETS),'--plan',str(PLAN)],check=True)
subprocess.run(base+['preflight','--plan',str(PLAN)],check=True)


In [ ]:
# Resumes by skipping only exact receipts that still validate.
subprocess.run(base+['run','--plan',str(PLAN),'--report',str(OUT/'capture-report.json')],check=True)


In [ ]:
subprocess.run(base+['check','--plan',str(PLAN),'--report',str(OUT/'verified-108-report.json')],check=True)
report=json.loads((OUT/'verified-108-report.json').read_text())
assert report['accepted']==report['target']==108
print('108 exact route receipts accepted. Download receipts, plan, report and evidence together.')
